In [3]:
# ==================== 用三个文档训练模型的完整代码 ====================
# 适用于 Jupyter Notebook，直接复制运行即可

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import numpy as np
import os
import json
import time
from pathlib import Path

print("="*60)
print("开始加载依赖...")
print("="*60)

# ==================== 1. 文档处理模块 ====================
class DocumentProcessor:
    """处理.docx和.txt文档"""
    
    @staticmethod
    def read_docx(file_path):
        """读取docx文件"""
        try:
            from docx import Document
            doc = Document(file_path)
            full_text = []
            for para in doc.paragraphs:
                if para.text.strip():
                    full_text.append(para.text)
            return '\n'.join(full_text)
        except ImportError:
            print("⚠️ python-docx未安装，跳过docx文件")
            return ""
        except Exception as e:
            print(f"❌ 读取docx失败 {file_path}: {e}")
            return ""
    
    @staticmethod
    def read_txt(file_path):
        """读取txt文件"""
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                return f.read()
        except UnicodeDecodeError:
            try:
                with open(file_path, 'r', encoding='gbk') as f:
                    return f.read()
            except:
                print(f"❌ 无法读取文件 {file_path}")
                return ""
    
    @classmethod
    def build_corpus(cls, file_paths, train_ratio=0.8):
        """从多个文件构建语料库"""
        all_texts = []
        file_info = []
        
        for path in file_paths:
            if not os.path.exists(path):
                print(f"⚠️ 文件不存在: {path}")
                continue
                
            if path.endswith('.docx'):
                text = cls.read_docx(path)
            elif path.endswith('.txt'):
                text = cls.read_txt(path)
            else:
                print(f"⚠️ 跳过未知格式: {path}")
                continue
            
            if text and len(text) > 100:
                all_texts.append(text)
                file_info.append({'path': path, 'length': len(text)})
                print(f"✓ 已加载: {path} ({len(text):,} 字符)")
        
        if not all_texts:
            raise ValueError("没有找到有效的文档文件")
        
        # 合并所有文本
        combined_text = '\n\n'.join(all_texts)
        
        # 保存完整语料
        with open('training_data.txt', 'w', encoding='utf-8') as f:
            f.write(combined_text)
        
        print(f"\n总字符数: {len(combined_text):,}")
        print(f"已保存到: training_data.txt")
        
        return combined_text, all_texts

# ==================== 2. 字符级Transformer模型 ====================
class CharTransformer(nn.Module):
    """字符级Transformer模型"""
    
    def __init__(self, vocab_size, d_model=128, nhead=8, num_layers=4, 
                 max_len=1024, dropout=0.1):
        super().__init__()
        
        self.vocab_size = vocab_size
        self.d_model = d_model
        
        # 基础组件
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = nn.Parameter(torch.randn(1, max_len, d_model))
        self.dropout = nn.Dropout(dropout)
        
        # Transformer编码器
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=512,
            batch_first=True, dropout=dropout
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # 输出层
        self.fc_out = nn.Linear(d_model, vocab_size)
    
    def forward(self, x, task='generate'):
        seq_len = min(x.shape[1], self.pos_encoding.shape[1])
        embedded = self.embedding(x)
        embedded = embedded + self.pos_encoding[:, :seq_len, :]
        embedded = self.dropout(embedded)
        out = self.transformer(embedded)
        logits = self.fc_out(out)
        return logits

# ==================== 3. 数据集类 ====================
class CharDataset(Dataset):
    """字符级数据集"""
    
    def __init__(self, text, seq_len=256, stride=None, char_to_idx=None):
        self.seq_len = seq_len
        self.stride = stride or seq_len // 2
        
        if char_to_idx is None:
            chars = sorted(list(set(text)))
            self.char_to_idx = {ch: i for i, ch in enumerate(chars)}
            self.idx_to_char = {i: ch for ch, i in self.char_to_idx.items()}
            self.vocab_size = len(chars)
        else:
            self.char_to_idx = char_to_idx
            self.idx_to_char = {i: ch for ch, i in char_to_idx.items()}
            self.vocab_size = len(char_to_idx)
        
        # 转为数字序列
        self.data = torch.tensor([self.char_to_idx.get(ch, 0) for ch in text], dtype=torch.long)
        
        # 创建序列
        self.xs = []
        self.ys = []
        for i in range(0, len(self.data) - seq_len - 1, self.stride):
            self.xs.append(self.data[i:i+seq_len])
            self.ys.append(self.data[i+1:i+seq_len+1])
        
        print(f"数据集创建完成: {len(self.xs)} 个样本, 词汇表大小: {self.vocab_size}")
    
    def __len__(self):
        return len(self.xs)
    
    def __getitem__(self, idx):
        return self.xs[idx], self.ys[idx]

# ==================== 4. 训练函数 ====================
def train_model(model, train_loader, val_loader, epochs=50, lr=0.001, device='cpu'):
    """训练模型"""
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.9)
    
    best_val_loss = float('inf')
    history = {'train_loss': [], 'val_loss': []}
    
    print(f"\n开始训练...")
    print(f"设备: {device}")
    print(f"训练批次数: {len(train_loader)}, 验证批次数: {len(val_loader)}")
    print("="*60)
    
    for epoch in range(epochs):
        # 训练
        model.train()
        total_train_loss = 0
        for batch_idx, (batch_x, batch_y) in enumerate(train_loader):
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            
            predictions = model(batch_x)
            loss = criterion(predictions.reshape(-1, model.vocab_size), batch_y.reshape(-1))
            
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            total_train_loss += loss.item()
            
            if (batch_idx + 1) % 50 == 0:
                print(f"Epoch {epoch+1}/{epochs} | Batch {batch_idx+1}/{len(train_loader)} | Loss: {loss.item():.4f}")
        
        # 验证
        model.eval()
        total_val_loss = 0
        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                predictions = model(batch_x)
                loss = criterion(predictions.reshape(-1, model.vocab_size), batch_y.reshape(-1))
                total_val_loss += loss.item()
        
        avg_train_loss = total_train_loss / len(train_loader)
        avg_val_loss = total_val_loss / len(val_loader)
        scheduler.step()
        
        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        
        # 保存最佳模型
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), 'best_model.pth')
            print(f"✓ 保存最佳模型 (loss: {avg_val_loss:.4f})")
        
        if (epoch + 1) % 5 == 0:
            print(f"\nEpoch {epoch+1}/{epochs} | 训练损失: {avg_train_loss:.4f} | 验证损失: {avg_val_loss:.4f}")
            print("-"*40)
    
    # 保存训练历史
    with open('training_history.json', 'w') as f:
        json.dump(history, f)
    
    print(f"\n训练完成！最佳验证损失: {best_val_loss:.4f}")
    
    return model, history

# ==================== 5. 生成函数 ====================
def generate_text(model, start_str, char_to_idx, idx_to_char, length=500, 
                  temperature=0.8, device='cpu'):
    """生成文本"""
    model.eval()
    
    # 转换起始字符串
    chars = [char_to_idx.get(ch, 0) for ch in start_str]
    input_ids = torch.tensor(chars).unsqueeze(0).to(device)
    generated = start_str
    
    print(f"\n生成中... (目标长度: {length} 字符)")
    
    for i in range(length):
        if input_ids.shape[1] > 512:
            input_ids = input_ids[:, -512:]
        
        with torch.no_grad():
            output = model(input_ids)
            logits = output[0, -1, :] / temperature
            probs = torch.softmax(logits, dim=-1)
            next_char_idx = torch.multinomial(probs, 1).item()
        
        next_char = idx_to_char[next_char_idx]
        generated += next_char
        input_ids = torch.cat([input_ids, torch.tensor([[next_char_idx]]).to(device)], dim=1)
        
        if (i + 1) % 100 == 0:
            print(f"已生成 {i+1}/{length} 字符...")
    
    return generated

# ==================== 6. 主训练流程 ====================
def main():
    print("="*60)
    print("开始训练字符级Transformer模型")
    print("="*60)
    
    # 1. 检查设备
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"使用设备: {device}")
    
    # 2. 文档路径 - 修改成你的文件路径
    doc_files = [
        '贵系.docx',      # 计算机科学文档
        '量化.docx',      # 量化投资文档  
        '一、 宇宙的基本框架与物理规律.docx'  # 宇宙学文档
    ]
    
    # 如果文件名有中文，确保路径正确
    # 如果你的文件在其他目录，加上完整路径，例如:
    # doc_files = [
    #     'C:/Users/你的用户名/Documents/贵系.docx',
    #     './data/量化.docx',
    # ]
    
    print("\n检查文档文件...")
    for f in doc_files:
        if os.path.exists(f):
            print(f"✓ 找到文件: {f}")
        else:
            print(f"❌ 文件不存在: {f}")
            print(f"   请确认文件路径是否正确")
            return
    
    # 3. 加载文档
    print("\n加载文档中...")
    processor = DocumentProcessor()
    combined_text, all_texts = processor.build_corpus(doc_files)
    
    print(f"\n总字符数: {len(combined_text):,}")
    print(f"文档数量: {len(all_texts)}")
    
    # 4. 创建数据集
    print("\n创建数据集...")
    dataset = CharDataset(combined_text, seq_len=256, stride=128)
    
    # 按文本划分训练/验证集（避免信息泄露）
    train_size = int(0.9 * len(dataset))
    val_size = len(dataset) - train_size
    train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])
    
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0)
    
    print(f"训练样本: {train_size}, 验证样本: {val_size}")
    
    # 5. 创建模型
    print("\n创建模型...")
    model = CharTransformer(
        vocab_size=dataset.vocab_size,
        d_model=256,      # 增大维度提高表现力
        nhead=8,
        num_layers=6,     # 更多层数
        max_len=1024,
        dropout=0.1
    )
    
    total_params = sum(p.numel() for p in model.parameters())
    print(f"模型参数量: {total_params:,}")
    print(f"词汇表大小: {dataset.vocab_size}")
    
    # 6. 训练
    model, history = train_model(
        model, train_loader, val_loader, 
        epochs=50, lr=0.001, device=device
    )
    
    # 7. 保存词汇表
    with open('vocab.json', 'w', encoding='utf-8') as f:
        json.dump({
            'char_to_idx': dataset.char_to_idx,
            'idx_to_char': {str(k): v for k, v in dataset.idx_to_char.items()}
        }, f, ensure_ascii=False)
    print("✓ 词汇表已保存到 vocab.json")
    
    # 8. 测试生成
    print("\n" + "="*60)
    print("测试文本生成")
    print("="*60)
    
    test_prompts = [
        "量子计算",
        "机器学习",
        "黑洞"
    ]
    
    for prompt in test_prompts:
        generated = generate_text(
            model, prompt, dataset.char_to_idx, dataset.idx_to_char,
            length=300, temperature=0.8, device=device
        )
        print(f"\n提示词: {prompt}")
        print(f"生成内容:\n{generated}")
        print("-"*60)
    
    print("\n✅ 训练完成！")
    print("模型已保存到: best_model.pth")
    print("词汇表已保存到: vocab.json")

# ==================== 7. 运行 ====================
if __name__ == "__main__":
    main()

开始加载依赖...
开始训练字符级Transformer模型
使用设备: cpu

检查文档文件...
✓ 找到文件: 贵系.docx
✓ 找到文件: 量化.docx
✓ 找到文件: 一、 宇宙的基本框架与物理规律.docx

加载文档中...
✓ 已加载: 贵系.docx (57,010 字符)
✓ 已加载: 量化.docx (20,752 字符)
✓ 已加载: 一、 宇宙的基本框架与物理规律.docx (33,532 字符)

总字符数: 111,298
已保存到: training_data.txt

总字符数: 111,298
文档数量: 3

创建数据集...
数据集创建完成: 868 个样本, 词汇表大小: 1899
训练样本: 781, 验证样本: 87

创建模型...
模型参数量: 4,398,955
词汇表大小: 1899

开始训练...
设备: cpu
训练批次数: 25, 验证批次数: 3
✓ 保存最佳模型 (loss: 6.1222)
✓ 保存最佳模型 (loss: 5.7909)
✓ 保存最佳模型 (loss: 5.0510)
✓ 保存最佳模型 (loss: 4.5469)
✓ 保存最佳模型 (loss: 4.2641)

Epoch 5/50 | 训练损失: 4.4626 | 验证损失: 4.2641
----------------------------------------
✓ 保存最佳模型 (loss: 4.0602)
✓ 保存最佳模型 (loss: 3.9188)
✓ 保存最佳模型 (loss: 3.8009)
✓ 保存最佳模型 (loss: 3.7141)
✓ 保存最佳模型 (loss: 3.6274)

Epoch 10/50 | 训练损失: 3.3970 | 验证损失: 3.6274
----------------------------------------
✓ 保存最佳模型 (loss: 3.5649)
✓ 保存最佳模型 (loss: 3.4743)
✓ 保存最佳模型 (loss: 3.3497)
✓ 保存最佳模型 (loss: 3.1114)
✓ 保存最佳模型 (loss: 2.6970)

Epoch 15/50 | 训练损失: 2.4252 | 验证损失: 2.6970
----------------

KeyboardInterrupt: 

In [10]:
import torch
import json
from model import CharTransformer  # 假设你的模型类在 model.py 里

# 1. 加载词汇表（训练时保存的）
with open('vocab.json', 'r', encoding='utf-8') as f:
    vocab = json.load(f)

char_to_idx = vocab['char_to_idx']
idx_to_char = {int(k): v for k, v in vocab['idx_to_char'].items()}
vocab_size = len(char_to_idx)

# 2. 重新创建模型（必须和训练时的架构一致）
model = CharTransformer(
    vocab_size=vocab_size,
    d_model=128,
    nhead=8,
    num_layers=4,
    max_len=512,
    dropout=0.1
)

# 3. 加载最佳模型权重
model.load_state_dict(torch.load('best_model.pth', map_location='cpu'))
model.eval()

# 4. 测试生成
def generate_text(model, start_str, length=200, temperature=0.8):
    """简化版生成函数"""
    chars = [char_to_idx.get(ch, 0) for ch in start_str]
    input_ids = torch.tensor(chars).unsqueeze(0)
    generated = start_str
    
    for _ in range(length):
        with torch.no_grad():
            output = model(input_ids)
            logits = output[0, -1, :] / temperature
            probs = torch.softmax(logits, dim=-1)
            next_char_idx = torch.multinomial(probs, 1).item()
        
        next_char = idx_to_char[next_char_idx]
        generated += next_char
        input_ids = torch.cat([input_ids, torch.tensor([[next_char_idx]])], dim=1)
        
        if input_ids.shape[1] > 512:
            input_ids = input_ids[:, -512:]
    
    return generated

# 5. 运行测试
prompt = "什么是图灵机？"
result = generate_text(model, prompt, length=200)
print("="*50)
print("生成结果：")
print("="*50)
print(result)

ModuleNotFoundError: No module named 'model'